# Baseline DINOv2-L + ALP + SAM-H — suporte K1

Notebook consolidado de `00_baseline_original.ipynb`. O original permanece como registro da exploração.

**Execute do início ao fim em um kernel novo do ambiente `protosam`.** Coloque este arquivo em `/workspace/ProtoSAM-repro/exp/`.

Modos na próxima célula:
- `consultar`: lê os resultados existentes, sem GPU ou carregamento de modelos (padrão).
- `verificar`: executa a primeira query do manifesto com ALP e SAM e compara as máscaras, pixel a pixel, com o baseline salvo. É a próxima verificação recomendada.
- `avaliar`: avalia as 100 queries em um diretório novo. Para retomar, informe seu caminho em `RETOMAR_DIR` e execute desde o início.

Não há downloads, treinamento, alteração dos arquivos oficiais ou sobrescrita da avaliação histórica. Novas execuções salvam máscaras compactas; não duplicam pesos nem todos os logits. Uma avaliação nova calcula ALP isolado e o pipeline completo separadamente, como na exploração; isso duplica a extração do ALP por query, sem cache de suporte.

O suporte K1 é fixo. Este protocolo não representa variabilidade entre suportes nem comprova reprodução das tabelas do artigo. Os tempos são diagnósticos, não benchmark.

In [ ]:
from pathlib import Path
MODO = "verificar"  # consultar | verificar | avaliar
RETOMAR_DIR = None  # caminho de uma execução DESTE notebook, somente em avaliar
REPO = Path("/workspace/ProtoSAM-repro")
HISTORICO = REPO / "reproducao/01_avaliacao_K1_20261004T140725337013Z"
COMPARACAO = HISTORICO / "comparacao_completa_alp_sam_20261006T083800723726Z"
BASE_OFICIAL = "64868c909668aac9ad517f7192a2712e88268e8f"
SUPORTE = "cju3xtufwiv9c0818djsc4cqd.png"
assert MODO in {"consultar", "verificar", "avaliar"}
assert RETOMAR_DIR is None or MODO == "avaliar"

## 1. Resultados e manifesto históricos

A lista e a ordem das queries vêm do manifesto salvo, não de uma nova seleção. As referências ALP e SAM são unidas pelo nome da imagem. Esta seção funciona sem carregar modelos.

In [ ]:
import os, sys, json, hashlib, subprocess, platform, importlib.metadata
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display
assert REPO.is_dir(), REPO
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
manifesto = pd.read_csv(HISTORICO / "manifesto_teste.csv")
historico_sam = pd.read_csv(HISTORICO / "por_imagem.csv").set_index("query")
historico_par = pd.read_csv(COMPARACAO / "por_imagem.csv").set_index("query")
assert len(manifesto) == 100 and manifesto["query"].is_unique
assert historico_sam.index.is_unique and historico_par.index.is_unique
assert set(manifesto["query"]) == set(historico_sam.index) == set(historico_par.index)
assert historico_sam["suporte"].eq(SUPORTE).all()
assert historico_par["suporte"].eq(SUPORTE).all()
for col in ("predicao_alp", "predicao_sam"):
    assert all(Path(p).is_file() for p in historico_par[col]), col
for metrica in ("Dice", "IoU", "precisao", "recall"):
    assert np.allclose(historico_par.loc[historico_sam.index, "SAM_" + metrica],
                       historico_sam[metrica], rtol=0, atol=1e-12)
metricas = ("Dice", "IoU", "precisao", "recall")
display(pd.DataFrame({modelo: {m: historico_par[f"{modelo}_{m}"].mean()
    for m in metricas} for modelo in ("ALP", "SAM")}))
delta = historico_par["SAM_Dice"] - historico_par["ALP_Dice"]
print("SAM melhora:", int((delta > 1e-6).sum()), "| piora:", int((delta < -1e-6).sum()))
print("Delta médio Dice:", float(delta.mean()))
print("Modo selecionado:", MODO)

## 2. Ambiente e configuração para inferência

Executada somente em `verificar` ou `avaliar`. Mantém FP32, semente 42, CCA desativada, pontos de confiança + centroide e caixa, `isval=True`, janela 2 e resolução do encoder 672. O canvas tem 1024×1024 e as métricas incluem padding.

Preserva o `image_size` como lista no wrapper, tal como na execução histórica. Não corrige aqui o metadado `original_size` nem a conversão min–max antes do SAM. O tamanho usado nas métricas vem dos tensores.

A comparação com a base oficial impede alterações nos arquivos rastreados dos modelos, datasets e utilitários. Notebooks novos são permitidos. O ambiente efetivo é registrado; semente fixa não garante identidade entre diferentes versões de bibliotecas/GPU.

In [ ]:
def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO, text=True).strip()

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def salvar_json(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)

def medir(pred, gt):
    tp, fp, fn = (int(x.sum()) for x in (pred & gt, pred & ~gt, ~pred & gt))
    return dict(TP=tp, FP=fp, FN=fn, Dice=2*tp/(2*tp+fp+fn+1e-8),
                IoU=tp/(tp+fp+fn+1e-8), precisao=tp/(tp+fp+1e-8), recall=tp/(tp+fn+1e-8))

if MODO != "consultar":
    import random, torch, cv2, inspect
    assert not git("diff", BASE_OFICIAL, "--", "models", "dataloaders", "util"), "Código diverge da base oficial."
    assert torch.cuda.is_available(), "CUDA indisponível"
    random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False
    torch.use_deterministic_algorithms(False)
    torch.set_float32_matmul_precision("highest")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = True
    DINO_REPO = REPO / "pretrained_model/hub/facebookresearch_dinov2_main"
    DINO_PESOS = REPO / "pretrained_model/hub/checkpoints/dinov2_vitl14_pretrain.pth"
    SAM_PESOS = REPO / "pretrained_model/sam_vit_h.pth"
    assert (DINO_REPO / "hubconf.py").is_file()
    assert DINO_PESOS.is_file() and SAM_PESOS.is_file()
    config_alp = dict(which_model="dinov2_l14", cls_name="grid_proto", proto_grid_size=8,
        lora=0, align=False, debug=False, use_coco_init=False, use_pos_enc=False,
        use_slice_adapter=False, adapter_layers=3, dinov2_loss=False)
    config_sam = dict(image_size=[1024,1024], sam_pretrained_path=str(SAM_PESOS),
        num_points_for_sam=1, use_points=True, use_bbox=True, use_mask=False,
        debug=False, use_cca=False, point_mode="both", use_sam_trans=True,
        coarse_pred_only=False, use_neg_points=False)
    ambiente = dict(python=sys.version, executavel=sys.executable, plataforma=platform.platform(),
        commit=git("rev-parse", "HEAD"), torch=str(torch.__version__), cuda=torch.version.cuda,
        cudnn=torch.backends.cudnn.version(), gpu=torch.cuda.get_device_name(0),
        threads=torch.get_num_threads(), threads_interop=torch.get_num_interop_threads(),
        cudnn_benchmark=torch.backends.cudnn.benchmark, tf32_matmul=torch.backends.cuda.matmul.allow_tf32,
        tf32_cudnn=torch.backends.cudnn.allow_tf32,
        bibliotecas={d.metadata["Name"]:d.version for d in importlib.metadata.distributions() if d.metadata["Name"]})
    print("GPU:", ambiente["gpu"], "| threads:", ambiente["threads"])

## 3. Dataset e suporte oficiais

Reconstrói os datasets com as transformações oficiais. Confere nomes, pares e caminhos contra o manifesto histórico. O suporte deve ter 51.940 pixels de primeiro plano, como no baseline.

In [ ]:
if MODO != "consultar":
    from dataloaders.PolypDataset import get_polyp_dataset
    from models.segment_anything.utils.transforms import ResizeLongestSide
    assert REPO in Path(inspect.getfile(get_polyp_dataset)).resolve().parents
    ds_suportes, ds_teste = get_polyp_dataset(image_size=(1024,1024), sam_trans=ResizeLongestSide(1024))
    assert len(ds_teste) == 100
    for item in manifesto.to_dict("records"):
        i = int(item["indice"])
        assert Path(ds_teste.images[i]).resolve() == Path(item["imagem"]).resolve()
        assert Path(ds_teste.gts[i]).resolve() == Path(item["mascara"]).resolve()
        assert Path(item["imagem"]).stem == Path(item["mascara"]).stem
    ids = [i for i,p in enumerate(ds_suportes.images) if Path(p).name == SUPORTE]
    assert len(ids) == 1
    assert Path(ds_suportes.gts[ids[0]]).stem == Path(SUPORTE).stem
    suporte = ds_suportes[ids[0]]
    assert int((suporte["label"] > .5).sum()) == 51940
    print("Suporte:", SUPORTE, "| índice:", ids[0])

## 4. Carregamento local dos modelos

A única adaptação no DINOv2 substitui o carregamento remoto por `source="local", pretrained=False` e carregamento estrito do checkpoint. O código oficial permanece no repositório. Nenhum otimizador é criado.

In [ ]:
if MODO != "consultar":
    import models.grid_proto_fewshot as modulo_alp
    from models.ProtoSAM import ProtoSAM, ALPNetWrapper, ALPNetInput
    assert Path(modulo_alp.__file__).resolve() == REPO / "models/grid_proto_fewshot.py"
    assert REPO in Path(inspect.getfile(ProtoSAM)).resolve().parents
    class FewShotSegDinoLocal(modulo_alp.FewShotSeg):
        def get_encoder(self):
            assert self.config["which_model"] == "dinov2_l14" and self.config["lora"] == 0
            self.encoder = torch.hub.load(str(DINO_REPO.resolve()), "dinov2_vitl14", source="local", pretrained=False)
            pesos = torch.load(DINO_PESOS, map_location="cpu", weights_only=True)
            print(self.encoder.load_state_dict(pesos, strict=True))
            lado = max(self.image_size // 14, modulo_alp.DEFAULT_FEATURE_SIZE)
            self.config["feature_hw"] = [lado, lado]
    assert "alpnet" not in globals(), "Modelos já carregados: continue na próxima célula ou reinicie o kernel."
    alpnet = FewShotSegDinoLocal(image_size=672, pretrained_path=None, cfg=config_alp.copy()).to("cuda", dtype=torch.float32)
    alpnet.eval().requires_grad_(False)
    wrapper = ALPNetWrapper(alpnet)
    protosam = ProtoSAM(coarse_segmentation_model=wrapper, **config_sam).to("cuda", dtype=torch.float32)
    protosam.eval().requires_grad_(False)
    alpnet.eval().requires_grad_(False)
    assert not any(p.requires_grad for p in alpnet.parameters())
    assert not any(p.requires_grad for p in protosam.sam.parameters())
    assert not alpnet.training and not protosam.training and not protosam.sam.training
    sup_img = suporte["image"].unsqueeze(0).cuda()
    sup_mask = suporte["label"].unsqueeze(0).cuda()
    print("Modelos congelados. Grade:", alpnet.config["feature_hw"])

In [ ]:
from datetime import datetime, timezone
from PIL import Image
import inspect
import numpy as np
import pandas as pd
import torch

assert alpnet.config["which_model"] == "dinov2_l14"
assert not alpnet.training and not protosam.training
assert protosam.use_cca is False
assert len(manifesto) == 100 and manifesto["query"].is_unique

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIR_COMPARACAO_DINO = REPO / "reproducao" / (
    f"dinov2_l14_k1_selecao_sam_exploratoria_{stamp}"
)
DIR_COMPARACAO_DINO.mkdir(parents=True, exist_ok=False)

for pasta in ("original", "maior_score", "registros"):
    (DIR_COMPARACAO_DINO / pasta).mkdir()

manifesto.to_csv(
    DIR_COMPARACAO_DINO / "manifesto_teste.csv", index=False
)
salvar_json(DIR_COMPARACAO_DINO / "config.json", {
    "tipo": "analise exploratoria pareada",
    "encoder": "dinov2_l14",
    "suporte": SUPORTE,
    "resolucao_encoder": 672,
    "config_alp": alpnet.config,
    "config_sam": config_sam,
    "ambiente": ambiente,
    "referencia_historica": str(HISTORICO),
    "regras": ["alternativa_0", "argmax_score_por_componente"],
    "pesos_sha256": {
        str(p.resolve()): sha256(p)
        for p in (DINO_PESOS, SAM_PESOS)
    },
})
(DIR_COMPARACAO_DINO / "selecao_original.py").write_text(
    inspect.getsource(type(protosam).predict_w_points_bbox),
    encoding="utf-8",
)

predict_original_dino = protosam.predictor.predict
acumulador_dino = {}

def observar_predict_dino(*args, **kwargs):
    saida = predict_original_dino(*args, **kwargs)
    mascaras, scores, _ = saida

    assert not kwargs.get("return_logits", False)
    mascaras = np.asarray(mascaras)
    scores = np.asarray(scores).reshape(-1)
    assert np.isfinite(scores).all()
    assert len(mascaras) == len(scores)
    assert np.isin(mascaras, [0, 1]).all()
    mascaras = mascaras.astype(bool)

    melhor = int(scores.argmax())
    acumulador_dino["original"] |= mascaras[0]
    acumulador_dino["maior_score"] |= mascaras[melhor]
    acumulador_dino["componentes"].append({
        "scores": scores.tolist(),
        "indice_maior_score": melhor,
    })
    return saida

def dice_comparacao(pred, gt):
    total = int(pred.sum()) + int(gt.sum())
    return 2 * int((pred & gt).sum()) / total if total else 1.0

linhas_dino = []
estado, erro = "executando", None

print("Diretório:", DIR_COMPARACAO_DINO)
print("GPU:", torch.cuda.get_device_name())

try:
    protosam.predictor.predict = observar_predict_dino

    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        for item in manifesto.sort_values("indice").to_dict("records"):
            indice = int(item["indice"])
            nome = item["query"]
            assert Path(ds_teste.images[indice]).name == nome

            exemplo = ds_teste[indice]
            gt = exemplo["label"].cpu().numpy() > 0.5

            # Confere a referência com os registros históricos.
            with Image.open(historico_sam.loc[nome, "predicao"]) as img:
                mascara_historica = np.asarray(img.convert("L")) > 0
            assert mascara_historica.shape == gt.shape
            metricas_ref = medir(mascara_historica, gt)
            for chave in ("TP", "FP", "FN"):
                assert metricas_ref[chave] == int(
                    historico_sam.loc[nome, chave]
                ), (nome, chave)

            acumulador_dino = {
                "original": np.zeros_like(gt, dtype=bool),
                "maior_score": np.zeros_like(gt, dtype=bool),
                "componentes": [],
            }

            qry = exemplo["image"].unsqueeze(0).to(sup_img.device)
            entrada = ALPNetInput(
                support_images=[sup_img],
                support_labels=[sup_mask],
                query_images=qry,
                isval=True,
                val_wsize=2,
                show_viz=False,
                supp_fts=None,
            )
            saida, _ = protosam(
                query_image=qry,
                coarse_model_input=entrada,
                degrees_rotate=0,
            )
            assert torch.isfinite(saida).all()
            assert ((saida == 0) | (saida == 1)).all()
            original = saida.cpu().numpy() > 0

            if acumulador_dino["componentes"]:
                assert np.array_equal(
                    original, acumulador_dino["original"]
                ), f"União reconstruída diferente: {nome}"
                alternativa = acumulador_dino["maior_score"]
            else:
                alternativa = original.copy()

            d_original = dice_comparacao(original, gt)
            d_alternativa = dice_comparacao(alternativa, gt)

            linha = {
                "indice": indice,
                "query": nome,
                "Dice_original": d_original,
                "Dice_maior_score": d_alternativa,
                "delta": d_alternativa - d_original,
                "componentes": len(acumulador_dino["componentes"]),
                "pixels_divergentes_historico": int(
                    np.count_nonzero(original != mascara_historica)
                ),
            }

            for pasta, mascara in (
                ("original", original),
                ("maior_score", alternativa),
            ):
                Image.fromarray(
                    mascara.astype(np.uint8) * 255
                ).save(DIR_COMPARACAO_DINO / pasta / nome)

            salvar_json(
                DIR_COMPARACAO_DINO / "registros" / f"{Path(nome).stem}.json",
                {**linha, "alternativas": acumulador_dino["componentes"]},
            )
            linhas_dino.append(linha)

            temporario = DIR_COMPARACAO_DINO / "por_imagem.tmp.csv"
            pd.DataFrame(linhas_dino).to_csv(temporario, index=False)
            temporario.replace(DIR_COMPARACAO_DINO / "por_imagem.csv")

            print(
                f"[{len(linhas_dino)}/100] {nome}: "
                f"original={d_original:.6f} "
                f"maior_score={d_alternativa:.6f} "
                f"delta={linha['delta']:+.6f}",
                flush=True,
            )

    estado = "concluida"

except BaseException as exc:
    estado, erro = "interrompida", repr(exc)
    raise

finally:
    protosam.predictor.predict = predict_original_dino
    salvar_json(DIR_COMPARACAO_DINO / "status.json", {
        "status": estado,
        "avaliadas": len(linhas_dino),
        "previstas": 100,
        "erro": erro,
    })

df_dino = pd.DataFrame(linhas_dino)
resumo_dino = {
    "n": len(df_dino),
    "Dice_original": float(df_dino["Dice_original"].mean()),
    "Dice_maior_score": float(df_dino["Dice_maior_score"].mean()),
    "delta_medio": float(df_dino["delta"].mean()),
    "delta_mediano": float(df_dino["delta"].median()),
    "melhora": int((df_dino["delta"] > 0).sum()),
    "piora": int((df_dino["delta"] < 0).sum()),
    "empate": int((df_dino["delta"] == 0).sum()),
}
salvar_json(DIR_COMPARACAO_DINO / "resumo.json", resumo_dino)
print(json.dumps(resumo_dino, indent=2))

## 5. Diretório e retomada

`verificar` usa somente a primeira query do manifesto (não selecionada por desempenho). `avaliar` usa todas. Cada nova execução recebe um diretório exclusivo. Para retomar após reiniciar o kernel, copie o caminho impresso para `RETOMAR_DIR` na primeira célula.

A retomada exige a mesma configuração, ambiente registrado, conteúdo dos dados, código DINOv2 e checkpoints. Os hashes são calculados lendo arquivos existentes; nenhum checkpoint é copiado.

In [ ]:
if MODO != "consultar":
    # Hashes de dados e código para vincular a retomada aos mesmos conteúdos.
    caminhos_dados = {str(Path(p).resolve()) for p in manifesto["imagem"]} | {str(Path(p).resolve()) for p in manifesto["mascara"]}
    caminhos_dados |= {str(Path(ds_suportes.images[ids[0]]).resolve()), str(Path(ds_suportes.gts[ids[0]]).resolve())}
    identidade = dict(versao_executor=1, modo=MODO, base=BASE_OFICIAL, suporte=SUPORTE,
        config_alp=config_alp, config_sam=config_sam, image_size=672, isval=True, val_wsize=2,
        seed=42, autocast=False, manifesto_sha256=sha256(HISTORICO / "manifesto_teste.csv"),
        dados={p:sha256(p) for p in sorted(caminhos_dados)},
        pesos={str(p.resolve()):sha256(p) for p in (DINO_PESOS,SAM_PESOS)},
        codigo_dino={str(p.relative_to(DINO_REPO)):sha256(p) for p in sorted(DINO_REPO.rglob("*.py"))},
        ambiente=ambiente)
    if "RUN_DIR" not in globals():
        if RETOMAR_DIR:
            RUN_DIR = Path(RETOMAR_DIR).resolve()
            assert RUN_DIR.is_dir()
            assert json.loads((RUN_DIR / "config.json").read_text()) == identidade, "Configuração/ambiente/conteúdo mudou."
        else:
            stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
            RUN_DIR = REPO / "reproducao" / f"baseline_k1_{MODO}_{stamp}"
            RUN_DIR.mkdir(parents=True, exist_ok=False)
            salvar_json(RUN_DIR / "config.json", identidade)
            manifesto.to_csv(RUN_DIR / "manifesto_teste.csv", index=False)
    assert json.loads((RUN_DIR / "config.json").read_text()) == identidade
    for pasta in ("alp", "sam"):
        (RUN_DIR / pasta).mkdir(exist_ok=True)
    print("Diretório para execução/retomada:", RUN_DIR)

## 6. Inferência e salvamento incremental

Cada linha só é registrada depois de salvar as duas máscaras. CSV e resumo usam substituição atômica; ao retomar, máscaras concluídas são conferidas por hash. Uma interrupção durante uma imagem faz apenas essa imagem ser refeita.

A saída ALP isolada usa o mesmo `ALPNetInput`. O pipeline SAM usa o `forward` oficial sem hooks, com rotação zero. Os tempos das duas chamadas são separados e não constituem benchmark. Referências vazias são registradas e ignoradas, conforme o protocolo anterior.

In [ ]:
def carregar_binaria(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    assert img is not None and set(np.unique(img)).issubset({0,255}), path
    return img > 0

if MODO != "consultar":
    import time, traceback
    selecionadas = manifesto.iloc[:1] if MODO == "verificar" else manifesto
    csv_path = RUN_DIR / "por_imagem.csv"
    linhas = pd.read_csv(csv_path).to_dict("records") if csv_path.exists() else []
    feitas = {r["query"] for r in linhas}
    assert len(feitas) == len(linhas) and feitas.issubset(set(selecionadas["query"]))
    for r in linhas:
        if r["status"] == "avaliada":
            for modelo in ("ALP", "SAM"):
                assert sha256(r[f"predicao_{modelo}"]) == r[f"sha256_{modelo}"]
    estado, erro = "em_andamento", None
    try:
        with torch.inference_mode(), torch.autocast("cuda", enabled=False):
            for item in selecionadas.to_dict("records"):
                nome = item["query"]
                if nome in feitas:
                    continue
                exemplo = ds_teste[int(item["indice"])]
                gt = exemplo["label"].cpu().numpy() > .5
                linha = dict(query=nome, indice=int(item["indice"]), suporte=SUPORTE)
                if not gt.any():
                    linha["status"] = "ignorada_sem_objeto"
                else:
                    # Garante a mesma referência da avaliação histórica.
                    ref_sam = carregar_binaria(historico_sam.loc[nome,"predicao"])
                    assert ref_sam.shape == gt.shape == (1024,1024)
                    mref = medir(ref_sam,gt)
                    for k in ("TP","FP","FN"):
                        assert mref[k] == int(historico_sam.loc[nome,k]), (nome,k)
                    qry = exemplo["image"].unsqueeze(0).cuda()
                    def entrada():
                        return ALPNetInput(support_images=[sup_img], support_labels=[sup_mask],
                            query_images=qry, isval=True, val_wsize=2, show_viz=False, supp_fts=None)
                    torch.cuda.synchronize(); inicio = time.perf_counter()
                    logits = wrapper(entrada())
                    torch.cuda.synchronize(); tempo_alp = time.perf_counter()-inicio
                    assert tuple(logits.shape) == (1,2,1024,1024) and torch.isfinite(logits).all()
                    pred_alp = logits.argmax(1)[0].cpu().numpy() > 0
                    del logits
                    torch.cuda.synchronize(); inicio = time.perf_counter()
                    pred, scores = protosam(query_image=qry, coarse_model_input=entrada(), degrees_rotate=0)
                    torch.cuda.synchronize(); tempo_sam = time.perf_counter()-inicio
                    pred = pred.cpu()
                    assert tuple(pred.shape) == (1024,1024) and torch.isfinite(pred).all()
                    assert ((pred == 0) | (pred == 1)).all()
                    pred_sam = pred.numpy() > 0
                    linha.update(status="avaliada", segundos_ALP=tempo_alp, segundos_pipeline=tempo_sam,
                        numero_scores_sam=len(scores), scores_sam=json.dumps([float(s) for s in scores]))
                    for modelo, mascara in (("ALP",pred_alp),("SAM",pred_sam)):
                        destino = RUN_DIR / modelo.lower() / nome
                        assert cv2.imwrite(str(destino), mascara.astype(np.uint8)*255)
                        linha[f"predicao_{modelo}"] = str(destino)
                        linha[f"sha256_{modelo}"] = sha256(destino)
                        linha.update({f"{modelo}_{k}":v for k,v in medir(mascara,gt).items()})
                        ref = carregar_binaria(historico_par.loc[nome,f"predicao_{modelo.lower()}"])
                        assert ref.shape == mascara.shape
                        linha[f"pixels_divergentes_{modelo}"] = int((ref != mascara).sum())
                    linha["delta_Dice"] = linha["SAM_Dice"] - linha["ALP_Dice"]
                linhas.append(linha)
                tmp = RUN_DIR / "por_imagem.tmp.csv"
                pd.DataFrame(linhas).to_csv(tmp,index=False); tmp.replace(csv_path)
                feitas.add(nome)
                mensagem = f"[{len(feitas)}/{len(selecionadas)}] {nome}: {linha['status']}"
                if linha["status"] == "avaliada":
                    mensagem += f" ALP={linha['ALP_Dice']:.6f} SAM={linha['SAM_Dice']:.6f}"
                print(mensagem,flush=True)
                with (RUN_DIR / "console.log").open("a",encoding="utf-8") as f:
                    f.write(mensagem + "\n")
        estado = "concluida"
    except BaseException as exc:
        estado, erro = "interrompida", repr(exc)
        with (RUN_DIR / "console.log").open("a",encoding="utf-8") as f:
            f.write(traceback.format_exc())
        raise
    finally:
        validos = [r for r in linhas if r["status"] == "avaliada"]
        resumo = dict(status=estado, previstas=len(selecionadas), avaliadas=len(validos),
            ignoradas_sem_objeto=len(linhas)-len(validos), pendentes=len(selecionadas)-len(linhas), erro=erro,
            medias_por_imagem={modelo:{m:float(np.mean([r[f"{modelo}_{m}"] for r in validos]))
                if validos else None for m in metricas} for modelo in ("ALP","SAM")})
        salvar_json(RUN_DIR / "resumo.json",resumo)
        print(json.dumps(resumo,indent=2,ensure_ascii=False))
    if MODO == "verificar":
        assert len(validos) == 1
        divergencias = {m:int(validos[0][f"pixels_divergentes_{m}"]) for m in ("ALP","SAM")}
        print("Pixels divergentes da referência:", divergencias)
        assert not any(divergencias.values()), "Diferenças detectadas; registre e investigue antes de avaliar as 100 queries."
        print("A primeira query reproduziu exatamente as duas máscaras históricas.")

## 7. Comparação e visualização

Em consulta, usa as máscaras históricas. Em inferência, usa somente linhas concluídas desta execução. O gráfico pareado inclui todas as imagens disponíveis; a figura qualitativa usa a primeira query do manifesto, sem seleção pelo Dice. As figuras são mostradas sem gravar novos arquivos em modo consulta.

In [ ]:
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from IPython.display import Image, display
from io import BytesIO
import cv2

def mostrar_figura(fig, nome):
    buffer = BytesIO()
    fig.savefig(buffer,format="png",dpi=130,bbox_inches="tight")
    if MODO != "consultar":
        (RUN_DIR / nome).write_bytes(buffer.getvalue())
    plt.close(fig)
    display(Image(data=buffer.getvalue()))

tabela = historico_par.copy() if MODO == "consultar" else pd.read_csv(RUN_DIR / "por_imagem.csv").query("status == 'avaliada'").set_index("query")
assert len(tabela), "Nenhuma imagem avaliada."
display(pd.DataFrame({m:{k:tabela[f"{m}_{k}"].mean() for k in metricas} for m in ("ALP","SAM")}))
d = tabela["SAM_Dice"]-tabela["ALP_Dice"]
print(f"n={len(tabela)} | melhora={(d>1e-6).sum()} | piora={(d < -1e-6).sum()} | empate={(d.abs()<=1e-6).sum()}")
print(f"Delta médio={d.mean():.6f} | mediano={d.median():.6f}")
fig,ax=plt.subplots(figsize=(5,5))
ax.scatter(tabela["ALP_Dice"],tabela["SAM_Dice"],alpha=.7)
ax.plot([0,1],[0,1],"--",color="gray")
ax.set(xlabel="Dice ALP",ylabel="Dice ALP + SAM",xlim=(0,1),ylim=(0,1),title=f"Comparação pareada — K1 (n={len(tabela)})")
mostrar_figura(fig,"dice_pareado.png")
nome = next(q for q in manifesto["query"] if q in tabela.index)
colunas = ("predicao_alp","predicao_sam") if MODO == "consultar" else ("predicao_ALP","predicao_SAM")
fig,axes=plt.subplots(1,2,figsize=(9,4))
for ax,col,modelo in zip(axes,colunas,("ALP","SAM")):
    mascara=carregar_binaria(tabela.loc[nome,col])
    ax.imshow(mascara,cmap="gray",vmin=0,vmax=1)
    ax.set_title(f"{modelo}: Dice={tabela.loc[nome,modelo+'_Dice']:.4f}")
    ax.axis("off")
fig.suptitle(nome)
mostrar_figura(fig,"primeira_query_mascaras.png")

## Encerramento e limites

Referência histórica (100 queries, K1): ALP Dice **0,6881575207991216**; ALP + SAM Dice **0,7247900719847806**; diferença média **+0,03663255118565908**. Houve melhora em 65 e piora em 35 imagens (tolerância 1e-6). Esses números são referência registrada, não saídas pré-executadas deste notebook.

As inspeções dos cinco piores casos e os escores intermediários permanecem no notebook original e no diretório `diagnostico_alp_sam_20261006T082641669776Z`. O baseline consolidado não introduz ajustes a partir dessas imagens.

Após `verificar` passar, este notebook serve de referência para o próximo experimento VMamba congelado. Migração do encoder, treinamento episódico, destilação, múltiplos suportes e benchmark de custo são etapas separadas.

**Validação da entrega:** estrutura e sintaxe verificadas fora do Runpod; inferência CUDA e igualdade numérica ainda precisam ser verificadas no ambiente do experimento.

In [ ]:
import sys
import json
import subprocess
import importlib.metadata
import torch

pacotes = {}
for nome in ("torch", "torchvision", "numpy", "Pillow", "timm", "xformers"):
    try:
        pacotes[nome] = importlib.metadata.version(nome)
    except importlib.metadata.PackageNotFoundError:
        pacotes[nome] = "não instalado"

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
)

ambiente_atual = {
    "python_executavel": sys.executable,
    "python_versao": sys.version,
    "pacotes": pacotes,
    "gpu_driver_memoria": gpu.stdout.strip() or gpu.stderr.strip(),
    "cuda_do_pytorch": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "threads": torch.get_num_threads(),
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
    "tf32_cudnn": torch.backends.cudnn.allow_tf32,
    "algoritmos_deterministicos": torch.are_deterministic_algorithms_enabled(),
    "precisao_matmul": torch.get_float32_matmul_precision(),
}

print(json.dumps(ambiente_atual, indent=2, ensure_ascii=False))

In [ ]:
from pathlib import Path

raiz = Path("/workspace/ProtoSAM-repro/reproducao")
pastas = sorted(
    p for p in raiz.glob("baseline_k1_verificar_*")
    if p.is_dir()
)

print(f"Verificações encontradas: {len(pastas)}")
for pasta in pastas:
    print(f"\n{pasta}")
    for arquivo in sorted(pasta.rglob("*")):
        if arquivo.is_file() and arquivo.suffix.lower() in (".csv", ".png"):
            print("  ", arquivo.relative_to(pasta))

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np

raiz = Path("/workspace/ProtoSAM-repro/reproducao")
execucao_1 = raiz / "baseline_k1_verificar_20261008T083806673655Z"
execucao_2 = raiz / "baseline_k1_verificar_20261008T084553903809Z"
query = "cju0u82z3cuma0835wlxrnrjv.png"

for modelo in ("alp", "sam"):
    with Image.open(execucao_1 / modelo / query) as img:
        mascara_1 = np.array(img)
    with Image.open(execucao_2 / modelo / query) as img:
        mascara_2 = np.array(img)

    assert mascara_1.shape == mascara_2.shape, (
        f"{modelo}: dimensões diferentes: "
        f"{mascara_1.shape} versus {mascara_2.shape}"
    )
    assert mascara_1.ndim == 2, f"{modelo}: máscara não está em formato 2D"

    divergentes = int(np.count_nonzero(mascara_1 != mascara_2))
    print(
        f"{modelo.upper()}: {divergentes} pixels divergentes "
        f"| máscaras idênticas: {divergentes == 0}"
    )

In [ ]:
import subprocess

repo_antigo = "/workspace/ProtoSAM"
base_oficial = "64868c909668aac9ad517f7192a2712e88268e8f"

comandos = [
    ("Branch atual", ["branch", "--show-current"]),
    ("Commit atual", ["rev-parse", "HEAD"]),
    ("Alterações locais", ["status", "--short"]),
    (
        "Arquivos alterados desde a base oficial",
        ["diff", "--name-status", base_oficial, "HEAD"],
    ),
]

for titulo, argumentos in comandos:
    resultado = subprocess.run(
        ["git", "-C", repo_antigo, *argumentos],
        capture_output=True,
        text=True,
    )
    print(f"\n=== {titulo} ===")
    print(resultado.stdout.strip() or "(sem saída)")
    if resultado.returncode != 0:
        raise RuntimeError(resultado.stderr.strip())

In [ ]:
import subprocess
from pathlib import Path

repo = Path("/workspace/ProtoSAM")
base = "64868c909668aac9ad517f7192a2712e88268e8f"

resultado = subprocess.run(
    [
        "git", "-C", str(repo),
        "diff", "--no-ext-diff", "--unified=8",
        base, "HEAD", "--", "models/grid_proto_fewshot.py",
    ],
    capture_output=True,
    text=True,
    check=True,
)
print(resultado.stdout)

arquivo = repo / "models/vmamba.py"
linhas = arquivo.read_text(encoding="utf-8").splitlines()

print(f"\n=== models/vmamba.py: {len(linhas)} linhas ===")
print("Imports e declarações de classes/funções:")
for numero, linha in enumerate(linhas, start=1):
    texto = linha.lstrip()
    if texto.startswith(("import ", "from ", "class ", "def ")):
        print(f"{numero:4d}: {linha}")

In [ ]:
from pathlib import Path

repo = Path("/workspace/ProtoSAM")
arquivo = repo / "models/vmamba.py"
linhas = arquivo.read_text(encoding="utf-8").splitlines()

trechos = [
    ("Disponibilidade de Triton", 1, 40),
    ("Seleção do backend selective scan", 779, 888),
    ("Configuração e execução do SS2D", 1385, 1572),
    ("Adaptação de chaves e carregamento do backbone", 2061, 2158),
]

for titulo, inicio, fim in trechos:
    print(f"\n=== {titulo} ===")
    for numero in range(inicio, min(fim, len(linhas)) + 1):
        print(f"{numero:4d}: {linhas[numero - 1]}")

checkpoint = repo / "pretrained_model/vmamba_tiny_v2.pth"
print("\n=== Checkpoint esperado ===")
print("Caminho:", checkpoint)
print("Existe:", checkpoint.is_file())
if checkpoint.is_file():
    print("Destino real:", checkpoint.resolve())
    print("Tamanho em MiB:", round(checkpoint.stat().st_size / 1024**2, 2))

In [ ]:
from pathlib import Path
from collections.abc import Mapping
import torch

caminho = Path(
    "/workspace/ProtoSAM/pretrained_model/vmamba_tiny_v2.pth"
)

checkpoint_vmamba = torch.load(
    caminho,
    map_location="cpu",
    weights_only=True,
)

print("Tipo do checkpoint:", type(checkpoint_vmamba).__name__)
assert isinstance(checkpoint_vmamba, Mapping), "Estrutura inesperada"

print("Chaves principais:", list(checkpoint_vmamba.keys())[:30])

if "model" in checkpoint_vmamba:
    pesos_vmamba = checkpoint_vmamba["model"]
elif "state_dict" in checkpoint_vmamba:
    pesos_vmamba = checkpoint_vmamba["state_dict"]
else:
    pesos_vmamba = checkpoint_vmamba

assert isinstance(pesos_vmamba, Mapping), "Pesos em formato inesperado"
assert all(isinstance(v, torch.Tensor) for v in pesos_vmamba.values()), (
    "O bloco selecionado não contém somente tensores"
)

print("Quantidade de tensores:", len(pesos_vmamba))

print("\nPrimeiras 12 chaves:")
for nome in list(pesos_vmamba)[:12]:
    print(nome, tuple(pesos_vmamba[nome].shape))

print("\nNormalização de saída e cabeça de classificação:")
for nome, tensor in pesos_vmamba.items():
    if (
        "outnorm" in nome
        or "classifier" in nome
        or nome in ("norm.weight", "norm.bias", "head.weight", "head.bias")
    ):
        print(nome, tuple(tensor.shape))

print("\nProjeções SSM do primeiro bloco:")
for nome, tensor in pesos_vmamba.items():
    if "layers.0.blocks.0." in nome and any(
        termo in nome for termo in ("x_proj", "dt_proj", "A_logs", "Ds")
    ):
        print(nome, tuple(tensor.shape))

In [ ]:
import importlib.util
import sys
from pathlib import Path
import torch

arquivo = Path("/workspace/ProtoSAM/models/vmamba.py")
nome_modulo = "vmamba_auditoria"

spec = importlib.util.spec_from_file_location(nome_modulo, arquivo)
vmamba_auditoria = importlib.util.module_from_spec(spec)
sys.modules[nome_modulo] = vmamba_auditoria
spec.loader.exec_module(vmamba_auditoria)

# pretrained=None evita o carregador antigo que captura exceções.
with torch.device("cpu"):
    encoder_auditoria = vmamba_auditoria.Backbone_VSSM(
        out_indices=(3,),
        pretrained=None,
        depths=[2, 2, 5, 2], dims=96, drop_path_rate=0.2,
        patch_size=4, in_chans=3, num_classes=1000,
        ssm_d_state=1, ssm_ratio=2.0, ssm_dt_rank="auto",
        ssm_act_layer="silu", ssm_conv=3, ssm_conv_bias=False,
        ssm_drop_rate=0.0, ssm_init="v0", forward_type="v05_noz",
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0,
        gmlp=False, patch_norm=True, norm_layer="ln2d",
        downsample_version="v3", patchembed_version="v2",
        use_checkpoint=False, posembed=False, imgsize=224,
    )

# strict=False é usado aqui apenas para revelar as incompatibilidades.
# Nenhuma exceção de carregamento será ocultada.
incompatibilidades = encoder_auditoria.load_state_dict(
    pesos_vmamba.copy(), strict=False
)
encoder_auditoria.eval().requires_grad_(False)

print("Chaves ausentes:", incompatibilidades.missing_keys)
print("Chaves inesperadas:", incompatibilidades.unexpected_keys)
print("Triton importado:", vmamba_auditoria.WITH_TRITON)
print(
    "Selective scan CUDA importado:",
    vmamba_auditoria.WITH_SELECTIVESCAN_MAMBA,
)

print("\nEstado de outnorm3 após o carregamento:")
for nome, tensor in encoder_auditoria.outnorm3.state_dict().items():
    print(
        nome,
        "shape:", tuple(tensor.shape),
        "mínimo:", tensor.min().item(),
        "máximo:", tensor.max().item(),
    )

In [ ]:
import torch

assert torch.cuda.is_available()
assert vmamba_auditoria.WITH_SELECTIVESCAN_MAMBA

contagem = {"cuda": 0, "fallback_torch": 0}
extensao = vmamba_auditoria.selective_scan_cuda
cuda_original = extensao.fwd
torch_original = vmamba_auditoria.selective_scan_torch

def contar_cuda(*args, **kwargs):
    contagem["cuda"] += 1
    return cuda_original(*args, **kwargs)

def contar_torch(*args, **kwargs):
    contagem["fallback_torch"] += 1
    return torch_original(*args, **kwargs)

# Gerador próprio: não modifica a semente global do notebook.
gerador = torch.Generator(device="cpu").manual_seed(42)
entrada_teste = torch.randn(1, 3, 224, 224, generator=gerador)

try:
    extensao.fwd = contar_cuda
    vmamba_auditoria.selective_scan_torch = contar_torch
    encoder_auditoria = encoder_auditoria.cuda().eval()

    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        saidas_teste = encoder_auditoria(entrada_teste.cuda())
        saida_cpu = saidas_teste[-1].cpu()
        del saidas_teste

    torch.cuda.synchronize()

finally:
    extensao.fwd = cuda_original
    vmamba_auditoria.selective_scan_torch = torch_original
    encoder_auditoria.cpu()

print("Chamadas:", contagem)
print("Shape de saída:", tuple(saida_cpu.shape))
print("Dtype:", saida_cpu.dtype)
print("Todos os valores finitos:", torch.isfinite(saida_cpu).all().item())

assert contagem["cuda"] > 0, "A extensão CUDA não foi chamada"
assert contagem["fallback_torch"] == 0, "Houve fallback para PyTorch"
assert tuple(saida_cpu.shape) == (1, 768, 7, 7)
assert torch.isfinite(saida_cpu).all().item()

In [ ]:
from pathlib import Path
import subprocess
import hashlib

origem = Path("/workspace/ProtoSAM")
destino = Path("/workspace/ProtoSAM-repro")
commit_origem = "a451fdfd832e4f1bd84259049e90efc606b03b25"
relativo = "models/vmamba.py"

def git(repo, *args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        check=True,
    ).stdout

branch = git(destino, "branch", "--show-current").decode().strip()
assert branch == "repro/vmamba-clean", f"Branch inesperada: {branch}"

# Obtém o arquivo do commit histórico identificado.
conteudo = git(origem, "show", f"{commit_origem}:{relativo}")
assert (origem / relativo).read_bytes() == conteudo, (
    "O arquivo local difere do commit inspecionado."
)

arquivo_destino = destino / relativo
if arquivo_destino.exists():
    assert arquivo_destino.read_bytes() == conteudo, (
        "Já existe um arquivo diferente no destino; nada foi sobrescrito."
    )
    print("Arquivo idêntico já presente.")
else:
    with arquivo_destino.open("xb") as arquivo:
        arquivo.write(conteudo)
    print("Arquivo copiado.")

assert arquivo_destino.read_bytes() == conteudo
print("Destino:", arquivo_destino)
print("Commit de origem:", commit_origem)
print("SHA256:", hashlib.sha256(conteudo).hexdigest())
print("\nStatus da branch limpa:")
print(git(destino, "status", "--short").decode())

In [ ]:
from pathlib import Path

raiz = Path("/workspace/ProtoSAM-repro/reproducao")

for pasta in sorted(raiz.glob("vmamba_small_k1_avaliar_*")):
    print(pasta)

In [ ]:
from pathlib import Path
import json
import pandas as pd

raiz = Path("/workspace/ProtoSAM-repro/reproducao")

execucoes = {
    "Tiny": raiz / "vmamba_tiny_k1_avaliar_20261008T095813008819Z",
    "Small": raiz / "vmamba_small_k1_avaliar_20261008T110134736074Z",
    "Base": raiz / "vmamba_base_k1_avaliar_20261008T134621894550Z",
}

linhas = []

for variante, pasta in execucoes.items():
    arquivo = pasta / "resumo.json"
    assert arquivo.is_file(), f"Resumo não encontrado: {arquivo}"

    resumo = json.loads(arquivo.read_text())
    assert resumo["n"] == resumo["previstas"] == 100, variante
    assert resumo["parcial"] is False, variante

    medias = resumo["medias_por_imagem"]
    linhas.append({
        "Encoder": variante,
        "N": resumo["n"],
        "Dice ALP": medias["ALP"]["Dice"],
        "Dice SAM": medias["SAM"]["Dice"],
        "IoU ALP": medias["ALP"]["IoU"],
        "IoU SAM": medias["SAM"]["IoU"],
        "Δ Dice": resumo["delta_medio"],
        "Melhora": resumo["melhora"],
        "Piora": resumo["piora"],
        "Empate": resumo["empate"],
    })

comparacao = pd.DataFrame(linhas).set_index("Encoder")
display(comparacao.round(6))

# Conferir o formato dos registros antes de montar a análise pareada.
base_por_imagem = pd.read_csv(execucoes["Base"] / "por_imagem.csv")
print("\nColunas de por_imagem.csv:")
print(base_por_imagem.columns.tolist())
display(base_por_imagem.head(2))

In [ ]:
from pathlib import Path
from PIL import Image
from io import BytesIO
import numpy as np

from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from IPython.display import display, Image as ImagemNotebook


def carregar_binaria(caminho):
    assert caminho.is_file(), f"Máscara não encontrada: {caminho}"
    with Image.open(caminho) as imagem:
        return np.asarray(imagem.convert("L")) > 0


queries = piores_base.index.tolist()

fig = Figure(figsize=(21, 3.5 * len(queries)))
canvas = FigureCanvasAgg(fig)
eixos = fig.subplots(len(queries), 7, squeeze=False)

for linha, query in enumerate(queries):
    nome = Path(query).stem + ".png"

    referencias = {
        variante: carregar_binaria(pasta / "gt" / nome)
        for variante, pasta in execucoes.items()
    }
    gt = referencias["Tiny"]

    for variante, mascara in referencias.items():
        assert np.array_equal(gt, mascara), (
            f"GT diferente: {variante}, {query}"
        )

    eixos[linha, 0].imshow(gt, cmap="gray", vmin=0, vmax=1)
    eixos[linha, 0].set_title(f"Referência\n{nome}", fontsize=8)

    coluna = 1
    for variante, pasta in execucoes.items():
        for etapa in ("ALP", "SAM"):
            mascara = carregar_binaria(
                pasta / etapa.lower() / nome
            )
            assert mascara.shape == gt.shape, (
                f"Shape diferente: {variante}/{etapa}, {query}"
            )

            painel = np.zeros((*gt.shape, 3), dtype=np.uint8)
            painel[mascara & gt] = [255, 255, 255]
            painel[mascara & ~gt] = [230, 65, 65]
            painel[~mascara & gt] = [65, 130, 255]

            dice = registros[variante].loc[query, f"{etapa}_Dice"]
            eixos[linha, coluna].imshow(painel)
            eixos[linha, coluna].set_title(
                f"{variante} — {etapa}\nDice: {dice:.4f}"
            )
            coluna += 1

    for eixo in eixos[linha]:
        eixo.axis("off")

fig.suptitle(
    "Cinco maiores quedas do Base após SAM\n"
    "Branco: objeto correto | Vermelho: falso positivo | "
    "Azul: objeto perdido",
    fontsize=14,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))

buffer = BytesIO()
canvas.print_png(buffer)
display(ImagemNotebook(data=buffer.getvalue()))
buffer.close()

print("GT idêntico entre as três variantes nos cinco casos.")



In [ ]:
import inspect
from models.ProtoSAM import ProtoSAM

print(inspect.getsource(ProtoSAM.forward))


In [ ]:
from pathlib import Path
import sys
import inspect

REPO = Path("/workspace/ProtoSAM-repro")
assert (REPO / "models" / "ProtoSAM.py").is_file(), (
    f"Arquivo não encontrado em {REPO}"
)

sys.path.insert(0, str(REPO))

from models.ProtoSAM import ProtoSAM

origem = Path(inspect.getfile(ProtoSAM)).resolve()
assert origem == (REPO / "models" / "ProtoSAM.py").resolve(), (
    f"Módulo carregado de outro local: {origem}"
)

print("Código carregado de:", origem)
print(inspect.getsource(ProtoSAM.forward))